# 04 - Linear regression with temporal backtesting

In [ ]:
import os, sys, pathlib
MODE = "real"            # "sample" shows the SYNTHETIC test run (clearly watermarked). Never mix the two.
os.environ["DATA_MODE"] = MODE
# os.environ["START_MONTH"] = "2024-06"; os.environ["END_MONTH"] = "2024-06"
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import json, duckdb, pandas as pd
from IPython.display import Image, Markdown, HTML, display
import config, provenance
print(provenance.banner())
if config.IS_SYNTHETIC:
    display(HTML("<h3 style='color:red'>SYNTHETIC DATA - pipeline test only, not Citi Bike findings</h3>"))
man = provenance.load()
assert man is not None, "No manifest for this run yet - run the earlier phases first."

In [ ]:
if not (config.REP_DATA / 'model_results.json').exists():
    import modeling, interpret
    # run: python src/modeling.py && python src/interpret.py  (modules execute on import in script mode)
res = json.loads((config.REP_DATA / 'model_results.json').read_text()); res['meta']

**Design:** rolling-origin expanding window (first 50% initial training, rest split into 4 test blocks). Model B leaves a gap equal to the forecast horizon and picks its station universe from the first half only. Baselines are fitted on training data only. Weather inputs are observed values standing in for forecasts.

In [ ]:
pd.DataFrame({m: {k: v['mean'] for k, v in d.items()} for m, d in res['duration_model']['summary'].items()}).T[['MAE','RMSE','R2','WAPE_%','MedianAE']].round(3)

In [ ]:
for k, b in res['demand_model'].items():
    print(k, '| best baseline:', b['best_baseline'], '| skill vs it: %+.1f%%' % (100*b['skill_mae_vs_best_baseline']))
    display(pd.DataFrame({m: {c: d[c]['mean'] for c in ['MAE','RMSE','WAPE_%','sMAPE_%','PoissonDeviance','bias']} for m, d in b['summary'].items()}).T.round(3))

**How to read this (guide, not a finding):** skill > 0 means the linear model beats the best baseline on MAE; check the fold-to-fold sd in `backtest_*.csv` before calling a difference real. R2 for duration is expected to be low.

In [ ]:
for n in ['12_duration_residuals','13_duration_error_analysis','14_duration_coefficients','15_demand_eval_h1','15_demand_eval_h24']: display(Image(str(config.FIG/f'{n}.png')))

## Auto-generated interpretation (numbers come from saved results)

In [ ]:
p = config.REPORT_DIR/'INTERPRETATION.md'
Markdown(p.read_text() if p.exists() else 'Run: python src/interpret.py')

## Your reading (write after viewing REAL results)
- Does OLS beat the best baseline, by how much, consistently across folds?
- Residual shape (heteroscedasticity, tails)?
- Where is error largest (hour, rider type, rain)?
- Does h=1 vs h=24 change the conclusion?